# part 2

In [1]:
import pandas as pd
import torch

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

In [15]:
df = pd.read_csv("insurance.csv")

print(df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [16]:
X = df.drop("charges", axis=1)
y = df["charges"]

In [17]:
numeric_features = ["age", "bmi"]
categorical_features = ["sex", "smoker", "region"]

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first"), categorical_features)
    ],
    remainder="passthrough"
)

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [20]:

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [21]:
mse = mean_squared_error(y_test, y_pred)

print("Mean Squared Error (MSE):", mse)

Mean Squared Error (MSE): 33596915.851361476


In [22]:
X_processed = preprocessor.fit_transform(X)

In [23]:
if hasattr(X_processed, "toarray"):
    X_processed = X_processed.toarray()

X_tensor = torch.FloatTensor(X_processed)
#y_tensor = torch.FloatTensor(y.values).reshape(-1, 1)
y_tensor = torch.FloatTensor(y.to_numpy().copy()).reshape(-1, 1)

print("\nTensor Shapes")
print("X:", X_tensor.shape)
print("y:", y_tensor.shape)


Tensor Shapes
X: torch.Size([1338, 8])
y: torch.Size([1338, 1])


In [24]:
import torch.nn as nn

In [25]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [26]:
if hasattr(X_train, "toarray"):
    X_train = X_train.toarray()
    X_test = X_test.toarray()


In [27]:
X_train = torch.FloatTensor(X_train)

X_test = torch.FloatTensor(X_test)

y_train = torch.FloatTensor(y_train.to_numpy().copy()).view(-1, 1)
y_test = torch.FloatTensor(y_test.to_numpy().copy()).view(-1, 1)

In [28]:
input_size = X_train.shape[1]

In [29]:
model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),

    nn.Linear(64, 64),
    nn.ReLU(),

    nn.Linear(64, 1)
)


In [30]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [31]:
epochs = 100

for epoch in range(epochs):

    outputs = model(X_train)
    loss = criterion(outputs, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {loss.item():.2f}")

Epoch [10/100] - Loss: 322171072.00
Epoch [20/100] - Loss: 320514080.00
Epoch [30/100] - Loss: 314945024.00
Epoch [40/100] - Loss: 301221920.00
Epoch [50/100] - Loss: 274346912.00
Epoch [60/100] - Loss: 231649488.00
Epoch [70/100] - Loss: 178746256.00
Epoch [80/100] - Loss: 134663840.00
Epoch [90/100] - Loss: 118954256.00
Epoch [100/100] - Loss: 115312656.00


In [32]:
model.eval()

with torch.no_grad():
    predictions = model(X_test)
    test_loss = criterion(predictions, y_test)

print(f"\nTest Loss (MSE): {test_loss.item():.2f}")


Test Loss (MSE): 116701480.00


## تحلیل نتایج

مقایسه عملکرد مدل رگرسیون خطی و شبکه عصبی

در این پروژه، عملکرد دو مدل رگرسیون خطی و شبکه عصبی برای پیش‌بینی هزینه‌های پزشکی مقایسه شد.

MSE مدل رگرسیون خطی: mse

Test Loss (MSE) شبکه عصبی: test_loss

اگر مقدار تست لاس شبکه عصبی کمتر از ام اس ای رگرسیون خطی باشد، نشان می‌دهد که شبکه عصبی توانسته روابط بین ویژگی‌های ورودی و هزینه‌های پزشکی را بهتر یاد بگیرد و دقت پیش‌بینی بالاتری داشته باشد. در غیر این صورت، اگر مقدار خطای شبکه عصبی بیشتر باشد، ممکن است به دلیل انتخاب نامناسب پارامترهای مدل، تعداد کم دوره‌های آموزش (اپک)، یا نیاز به تنظیم بهتر معماری شبکه باشد.

دلیل استفاده از تابع فعال‌ساز ReLU

تابع فعال‌ساز ReLU (Rectified Linear Unit)

ReLU(x)=max(0,x)

این تابع مقادیر منفی را به صفر تبدیل کرده و مقادیر مثبت را بدون تغییر عبور می‌دهد.

استفاده از تابع فعال سازی باعث می‌شود شبکه عصبی بتواند روابط غیرخطی را یاد بگیرد. اگر بین لایه‌های شبکه از تابع فعال‌ساز استفاده نشود، حتی با وجود چندین لایه، کل شبکه رفتاری مشابه یک مدل خطی خواهد داشت و قادر به مدل‌سازی روابط پیچیده نخواهد بود.

در این مسئله، هزینه‌های پزشکی به عواملی مانند سن، شاخص توده بدنی ، وضعیت استعمال دخانیات، تعداد فرزندان و منطقه سکونت وابسته است و رابطه این عوامل با هزینه‌ها کاملاً خطی نیست. برای مثال، اثر سیگار کشیدن بر هزینه درمان بسیار بیشتر از سایر عوامل است و ممکن است با افزایش سن یا شاخص توده بدنی به‌صورت غیرخطی تغییر کند. استفاده از تابع فعال سازی این امکان را فراهم می‌کند که شبکه چنین الگوهای پیچیده‌ای را یاد گرفته و پیش‌بینی دقیق‌تری نسبت به مدل رگرسیون خطی ارائه دهد.